In [ ]:
from pathlib import Path
import pandas as pd
from collections import defaultdict
import numpy as np
from rdkit import Chem
import pickle 
from quartz.data import MoleculeData
from quartz.grids import create_grid, to_dxgrid

In [ ]:
# Linux paths
# Load PDBs from directory and map to Excel Data
ligands_dir = Path("~/Documents/Major/Ligands/all_ligs").expanduser()
charges_dir = Path("sample_data/FXa_PDB_RESP_charges")
excel_path = Path(
    "~/Documents/Major/Ligands/full_database_ligands.xlsx"
).expanduser()

In [ ]:
# Load PDBs from directory and map to Excel Data
ligands_dir = Path("~/Documents/Uni/Major/Ligands/all_ligs").expanduser()
charges_dir = Path("sample_data/FXa_PDB_RESP_charges")
excel_path = Path(
    "~/Documents/Uni/Major/Ligands/full_database_ligands.xlsx"
).expanduser()


In [ ]:
# Read Excel file
print(f"Reading Excel file: {excel_path}")
fx_data = pd.read_excel(excel_path)
# Clean column names by stripping whitespace
fx_data.columns = fx_data.columns.str.strip()

pdb_files = list(ligands_dir.glob("*_out_reduced.pdb"))

mol_data_list = []

print(f"Found {len(pdb_files)} PDB files in {ligands_dir}")

for pdb_file in pdb_files:
    # Extract PDB ID from filename (e.g. 1EZQ_out_reduced.pdb -> 1EZQ)
    pdb_id = pdb_file.name.split("_")[0]
    print(f"Processing PDB ID: {pdb_id}")

    # Path to charge file
    charge_file = charges_dir / f"{pdb_id}_out_reduced.csv"
    # print(f"Looking for charge file: {charge_file}")

    if not charge_file.exists():
        print(f"Charge file not found for {pdb_id}, skipping...")
        continue

    extra = defaultdict(float)

    with open(pdb_file, "r") as f:
        pdb_content = f.readlines()
        for line in pdb_content:
            # print(line)
            if line.startswith("REMARK"):
                # print(line.strip())
                if "minimizedAffinity" in line:
                    extra["minimizedAffinity"] = float(line.split()[-1])
                    # print("Minimized Affinity:", extra['minimizedAffinity'])
                if "CNNscore" in line:
                    extra["CNNscore"] = float(line.split()[-1])
                    # print("CNN Score:", extra['CNNscore'])
                if "CNNaffinity" in line:
                    extra["CNNaffinity"] = float(line.split()[-1])
                    # print("CNN Affinity:", extra['CNNaffinity'])

            if (
                extra["minimizedAffinity"]
                and extra["CNNscore"]
                and extra["CNNaffinity"]
            ):
                break

    # Get pKi and Ki for the PDB ID
    # Assuming column is 'PDB ID' after stripping
    row = fx_data[fx_data["PDB ID"].astype(str).str.strip() == pdb_id]

    if row.empty:
        print(f"No FX Data found for {pdb_id}")
        continue

    # Handle potential duplicates or just take first
    ki = float(row["Ki (nM)"].values[0])
    # print("Ki:", ki)
    if "pKi" in row.columns:
        pki = float(row["pKi"].values[0])
    else:
        pki = -np.log10(ki * 1e-9)  # Convert Ki in nM to pKi
    # print("pKi:", pki)

    # Load Mol from PDB
    mol = Chem.MolFromPDBFile(str(pdb_file), removeHs=False)
    if mol is None:
        print(f"Failed to load PDB: {pdb_file.name}")
        continue


    if not charge_file.exists():
        print(f"Charge file not found for {pdb_id}")
        continue

    charges_df = pd.read_csv(charge_file)

    # Check atom count and charge assignment
    if len(charges_df) != mol.GetNumAtoms():
        print(
            f"Atom count mismatch for {pdb_id}: mol {mol.GetNumAtoms()} vs csv {len(charges_df)}"
        )
        continue

    # Assign RESP charges
    charge_sum = 0.0
    for idx, row_charge in charges_df.iterrows():
        atom = mol.GetAtomWithIdx(idx)
        resp_charge = float(row_charge["RESP"])
        atom.SetProp("RESP", str(resp_charge))
        charge_sum += resp_charge

        # Verify assignment
        if abs(float(atom.GetProp("RESP")) - resp_charge) > 1e-6:
            print(f"Charge assignment failed for {pdb_id} atom {idx}")

    # print(f"Processed {pdb_id}: Total Charge = {charge_sum:.4f}")

    # Create MoleculeData
    md = MoleculeData(
        atomic_numbers=np.array([a.GetAtomicNum() for a in mol.GetAtoms()]),
        coordinates=mol.GetConformer().GetPositions(),
        res={},
        smiles=Chem.MolToSmiles(mol),
        rdmol=mol,
        pdb_code=pdb_id,
        filename=pdb_id,
        ki=ki,
        pki=pki,
        extra=extra,
    )
    mol_data_list.append(md)

print(f"Prepared {len(mol_data_list)} MoleculeData objects")

In [ ]:
# Generate grids
print("Generating grids...")
grids = create_grid(mol_data_list, props=["RESP"], size=0.5, threshold=0.0001)

In [ ]:
# Save to pkl
output_file = "grids.pkl"
with open(output_file, "wb") as f:
    pickle.dump(grids, f)

In [ ]:
# Save to DX
print("Saving to DX format...")
to_dxgrid(grids)